# Oslo Børs company eligibility

Read the original export, remove exchange closures, and save eligible company rows to `Data/eligible`.
Run with `pandas` and `exchange_calendars==4.13.2`.

## Filter dates

Exclude XOSL non-trading dates over the last 15 years. Use the dates that remain in the last ten years to measure coverage. [XOSL calendar](https://github.com/gerrymanoim/exchange_calendars/blob/4.13.2/exchange_calendars/exchange_calendar_xosl.py) · [Euronext holiday calendars](https://www.euronext.com/en/trading/trading-hours-holidays) · [2020 year-end notice](https://connect2.euronext.com/sites/default/files/2020-11/IF201124CADE%202020%20Updated%20Holiday%20Calendar.pdf)

In [1]:
from pathlib import Path
from tempfile import NamedTemporaryFile

import exchange_calendars as xcals
import numpy as np
import pandas as pd

data_dir = Path("Data") if Path("Data").is_dir() else Path("../Data")
source = data_dir / "BMG9156K1018_10.09.2026_14.59.34.csv"
output = data_dir / "eligible" / f"{source.stem}_trading_days_eligible.csv"
identifiers = ["CompanyId", "Internal code"]
columns = ["Date", "ISIN", "AdjustedPrice", *identifiers]
rows = pd.read_csv(source, sep=";", usecols=columns, dtype="string")
rows["Date"] = pd.to_datetime(rows["Date"], format="%Y-%m-%d")

end = rows["Date"].max()
calendar_start = end - pd.DateOffset(years=15)
window_start = end - pd.DateOffset(years=10)
calendar = xcals.get_calendar("XOSL", start=calendar_start, end=end)
non_trading_days = pd.date_range(calendar_start, end).difference(calendar.sessions)
on_closed_day = rows["Date"].isin(non_trading_days)

history = rows.loc[~on_closed_day]
window = history.loc[history["Date"].between(window_start, end)]
trading_dates = pd.DatetimeIndex(window["Date"].unique()).sort_values()

print(f"Excluded {on_closed_day.sum():,} rows on {rows.loc[on_closed_day, 'Date'].nunique()} closure dates.")
print(f"Coverage window: {window_start.date()} to {end.date()} ({len(trading_dates):,} observed dates).")

Excluded 8,268 rows on 30 closure dates.
Coverage window: 2016-09-08 to 2026-09-08 (2,510 observed dates).


## Link ISINs and check coverage

Link ISINs transitively by nonmissing `CompanyId` or `Internal code` across the filtered history. Count a date once per company when any linked ISIN has a numeric `AdjustedPrice`. Require less than 95% missing dates and a longest gap of at most five trading dates, including at the window boundaries.

In [2]:
parent = {isin: isin for isin in history["ISIN"].unique()}


def company_root(isin: str) -> str:
    """Return the representative ISIN of a linked group."""
    while isin != parent[isin]:
        parent[isin] = parent[parent[isin]]
        isin = parent[isin]
    return isin


for column in identifiers:
    first_isin_by_id = {}
    links = history[["ISIN", column]].dropna().drop_duplicates()
    for isin, identifier in links.itertuples(index=False, name=None):
        first_isin = first_isin_by_id.setdefault(identifier, isin)
        parent[company_root(isin)] = company_root(first_isin)

company_by_isin = {isin: company_root(isin) for isin in parent}
company = window["ISIN"].map(company_by_isin)
prices = pd.to_numeric(window["AdjustedPrice"].str.replace(",", ".", regex=False), errors="coerce")
has_price = np.isfinite(prices.to_numpy(dtype=float, na_value=np.nan))
company_dates = pd.DataFrame({
    "Company": company[has_price],
    "Session": trading_dates.get_indexer(window.loc[has_price, "Date"]),
}).drop_duplicates().sort_values("Session")

n_dates = len(trading_dates)
eligible_companies = set()
for company_id, positions in company_dates.groupby("Company")["Session"]:
    missing_dates = n_dates - len(positions)
    # The endpoints include gaps before the first and after the last observation.
    missing_runs = np.diff(np.r_[-1, positions.to_numpy(), n_dates]) - 1
    if 100 * missing_dates < 95 * n_dates and missing_runs.max() <= 5:
        eligible_companies.add(company_id)

eligible_rows = window.loc[company.isin(eligible_companies)]
print(f"Eligible: {len(eligible_companies):,} companies, "
      f"{eligible_rows['ISIN'].nunique():,} ISINs, {len(eligible_rows):,} rows.")

Eligible: 115 companies, 140 ISINs, 298,839 rows.


## Export eligible rows

Copy retained records directly to preserve source values, formatting, order and overlapping share classes.

In [3]:
output.parent.mkdir(exist_ok=True)
keep_row = rows.index.isin(eligible_rows.index)

with NamedTemporaryFile(dir=output.parent, suffix=".tmp", delete=False) as dst:
    temporary = Path(dst.name)
    try:
        with source.open("rb") as src:
            dst.write(src.readline())
            for line, keep in zip(src, keep_row, strict=True):
                if keep:
                    dst.write(line)
        dst.close()
        temporary.replace(output)
    finally:
        temporary.unlink(missing_ok=True)

print(f"Wrote {output}")

Wrote ../Data/eligible/BMG9156K1018_10.09.2026_14.59.34_trading_days_eligible.csv
